In [1]:
# -*- coding: utf-8 -*-
%load_ext autoreload
%autoreload 2

In [1]:
! pip install json-repair

In [66]:
import json, json_repair
def extract_jsons(response, quite=False):    
    if isinstance(response, (dict, list)):
        # return as it is 
        # if not quite: print("extract_json", "response is already in json format")
        return response       
    elif isinstance(response, str):
        # Method 1
        try:
            # try simple to load it as json
            res = json.loads(response)
            # if not quite: print("extract_json", "response is already in jsons format")
            return res
        except:
            pass
            # if not quite: print("extract_json: simple json load failed. Trying to fix json string ...")
           
        # Method 2 
        try:
            # if not quite: print("extract_json", "response is not in json format. Trying to extract json from response")
            if '```json' in text:                
                out = text.split('```json')[1].split('```')[0].replace('\n','')
            elif '```' in text:
                out = text.split('```')[1].split('```')[0].replace('\n','')
            else:
                out = text

            res = json.loads(out)
            return res        
        except Exception as e:
            # if not quite: print(f"extract_json: unable to fix json string. Trying with json_repair ...")
            pass         
            # it is not in json string format
            
            # Method 3
            text = response
            try:                
                res = json_repair.loads(text)
                if isinstance(res, (dict, list)):
                    # if not quite: print("extract_json: result obtained using repair json")
                    return res
            except:
                if not quite: print("extract_json: unable to repair json string using json_repair. Raise exception")
                raise
    else:
        # if not quite: print("extract_json", "response is not a string or a dictionary")
        return {}  
    

In [57]:
response = """
    Overall, you have demonstrated qualities that align with the role's requirements outlined in the job description. Your performance was Good.

    { "evaluation": { "percentage": "80%", "message": "Good" } }

    I would like to offer you the opportunity for a follow-up interview. If you agree, we can conduct a new round of fresh, diverse questions, including more technical questions related to the specific position you are applying for. Would you like to proceed?
"""
val = extract_json(response)

In [54]:
import re

In [71]:
import json
import json_repair

def extract_json(response, quite=False):    
    if isinstance(response, (dict, list)):
        return response, ""  # Return as it is       
    elif isinstance(response, str):
        # Initialize variables
        json_part = None
        
        # Method 1: Try simple JSON load
        try:
            json_part = json.loads(response)
            return json_part, ""  # Return if already valid JSON with empty text
        except:
            pass
        
        # Method 2: Attempt to extract JSON from response
        try:
            # Find the first occurrence of '{' and the last occurrence of '}'
            start_index = response.index('{')
            end_index = response.rindex('}') + 1
            json_str = response[start_index:end_index]
            json_part = json.loads(json_str)
            # Remove the JSON part from the original response
            non_json_part = response.replace(json_str, '').strip()
            return json_part, non_json_part        
        except Exception:
            pass
            
        # Method 3: Try using json_repair
        try:                
            repaired_json = json_repair.loads(response)
            if isinstance(repaired_json, (dict, list)):
                repaired_json_str = json.dumps(repaired_json)  # Convert to string
                non_json_part = response.replace(repaired_json_str, '').strip()  # Remove JSON part from original response
                return repaired_json, non_json_part
        except:
            if not quite:
                print("extract_json: unable to repair json string using json_repair. Raise exception")
            raise

    # If no valid JSON was found, return None for both
    return None, None

# Example usage
response = """
    Overall, you have demonstrated qualities that align with the role's requirements outlined in the job description. Your performance was Good.

    

    I would like to offer you the opportunity for a follow-up interview. If you agree, we can conduct a new round of fresh, diverse questions, including more technical questions related to the specific position you are applying for. Would you like to proceed?
"""

json_output, other_text = extract_json(response)
print("JSON Output:", json_output)
print("Other Text:", other_text)

JSON Output: None
Other Text: None
